# Pokémon Analytics

Exploratory analysis over the dbt mart tables built from `PokemonData.csv`.

**Database:** `../dbt/data_stack.duckdb`  
**Marts available:**
- `main_marts.mart_pokemon_by_type` — avg stats per primary type
- `main_marts.mart_pokemon_by_generation` — stat trends across generations
- `main_marts.mart_legendary_comparison` — legendary vs non-legendary breakdown

Run `dbt run --profiles-dir .` from the `dbt/` directory before using this notebook.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.1f}".format)

DB_PATH = "../dbt/data_stack.duckdb"
con = duckdb.connect(DB_PATH, read_only=True)

def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).df()

print("Connected to", DB_PATH)

## 1. Raw data preview

A quick look at the cleaned staging layer before aggregation.

In [ ]:
pokemon = q("select * from main_intermediate.int_pokemon_stats order by base_stat_total desc")
print(f"{len(pokemon)} rows")
pokemon.head(10)

## 2. Stats by primary type

Which types have the strongest average base stat totals?

In [ ]:
by_type = q("select * from main_marts.mart_pokemon_by_type")
by_type

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.barh(by_type["primary_type"], by_type["avg_base_stat_total"], color="steelblue")
ax.invert_yaxis()
ax.set_xlabel("Avg Base Stat Total")
ax.set_title("Average Base Stat Total by Primary Type")
ax.xaxis.set_minor_locator(ticker.MultipleLocator(25))
ax.grid(axis="x", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

## 3. Stat profile by type

How do offensive vs defensive stat profiles differ across types?

In [ ]:
stats = ["avg_hp", "avg_attack", "avg_defense", "avg_special_attack", "avg_special_defense", "avg_speed"]
labels = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]

plot_df = by_type.set_index("primary_type")[stats]
plot_df.columns = labels

fig, ax = plt.subplots(figsize=(14, 7))
plot_df.plot(kind="bar", ax=ax, width=0.8)
ax.set_xlabel("")
ax.set_ylabel("Average Stat")
ax.set_title("Stat Profile by Primary Type")
ax.legend(loc="upper right", ncol=3)
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

## 4. Trends across generations

Did Pokémon get stronger over time? How did legendary density change?

In [ ]:
by_gen = q("select * from main_marts.mart_pokemon_by_generation")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Avg base stat total per generation
axes[0].plot(by_gen["generation"], by_gen["avg_base_stat_total"], marker="o", color="steelblue")
axes[0].set_title("Avg Base Stat Total by Generation")
axes[0].set_xlabel("Generation")
axes[0].set_ylabel("Avg Base Stat Total")
axes[0].set_xticks(by_gen["generation"])
axes[0].grid(linestyle="--", alpha=0.4)

# Legendary % per generation
axes[1].bar(by_gen["generation"], by_gen["legendary_pct"], color="goldenrod")
axes[1].set_title("Legendary Pokémon % by Generation")
axes[1].set_xlabel("Generation")
axes[1].set_ylabel("% of roster")
axes[1].set_xticks(by_gen["generation"])
axes[1].grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

by_gen

## 5. Legendary vs non-legendary

How much stronger are legendaries across every stat?

In [ ]:
leg = q("select * from main_marts.mart_legendary_comparison")

stat_cols = ["avg_hp", "avg_attack", "avg_defense", "avg_special_attack", "avg_special_defense", "avg_speed"]
stat_labels = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]

legendary_vals    = leg.loc[leg["is_legendary"] == True,  stat_cols].values[0]
nonlegendary_vals = leg.loc[leg["is_legendary"] == False, stat_cols].values[0]

x = range(len(stat_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i - width/2 for i in x], nonlegendary_vals, width, label="Non-Legendary", color="steelblue")
ax.bar([i + width/2 for i in x], legendary_vals,    width, label="Legendary",     color="goldenrod")

ax.set_xticks(list(x))
ax.set_xticklabels(stat_labels)
ax.set_ylabel("Average Stat")
ax.set_title("Legendary vs Non-Legendary — Avg Stats")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

leg

## 6. Ad-hoc queries

Use this cell to write your own queries against any layer.

In [ ]:
q("""
select
    pokemon_name,
    primary_type,
    secondary_type,
    base_stat_total
from main_intermediate.int_pokemon_stats
where is_legendary = false
order by base_stat_total desc
limit 10
""")